In [ ]:
# MarketMind Intelligence Platform V1
# Author: Sharique Mohammad
# Date: April 23, 2026

# MarketMind V1 - Data Exploration

**Notebook:** 01_data_exploration.ipynb  
**Created:** April 23, 2026  
**Purpose:** Explore all Gold layer tables, verify data loading, basic statistics

---

## Overview

This notebook explores the data loaded into the Gold layer:
- **gold.ohlcv_bars** - Market OHLCV data
- **gold.macro_indicators** - US economic indicators
- **gold.sec_filings** - SEC filing metadata
- **gold.corporate_actions** - Stock splits and dividends

Expected Data:
- OHLCV: 1,240 records (20 tickers × 62 trading days Q1 2026)
- Macro: 2,594 records (1970s-2025 historical)
- SEC Filings: 700 records (3 companies, 2022-2026)
- Corporate Actions: 0 records (none in period)

In [ ]:
# Import libraries
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 1. Database Connection

In [ ]:
# PostgreSQL connection parameters
DB_CONFIG = {
    'host': '172.31.32.1',
    'port': 5432,
    'database': 'marketmind_v1',
    'user': 'postgres',
    'password': '0940'
}

# Test connection
try:
    conn = psycopg2.connect(**DB_CONFIG)
    cur = conn.cursor()
    cur.execute('SELECT version()')
    version = cur.fetchone()[0]
    print("Database connection: SUCCESS")
    print(f"PostgreSQL version: {version.split(',')[0]}")
    conn.close()
except Exception as e:
    print(f"Database connection: FAILED - {e}")

## 2. Table Record Counts

In [ ]:
# Query all Gold table record counts
conn = psycopg2.connect(**DB_CONFIG)

tables = ['ohlcv_bars', 'macro_indicators', 'sec_filings', 'corporate_actions']
counts = {}

for table in tables:
    query = f'SELECT COUNT(*) FROM gold.{table}'
    df = pd.read_sql(query, conn)
    counts[table] = df.iloc[0, 0]

conn.close()

# Display as DataFrame
count_df = pd.DataFrame(list(counts.items()), columns=['Table', 'Record Count'])
count_df['Record Count'] = count_df['Record Count'].apply(lambda x: f'{x:,}')

print("\nGold Layer Record Counts:")
print("=" * 50)
print(count_df.to_string(index=False))
print("=" * 50)

In [ ]:
# Visualize record counts as bar chart
fig, ax = plt.subplots(figsize=(10, 6))
count_vals = [counts[t] for t in tables]
colors = ["steelblue", "darkorange", "green", "crimson"]
bars = ax.bar(tables, count_vals, color=colors, edgecolor="black", linewidth=1.2)

ax.set_xlabel("Gold Table", fontsize=12, fontweight="bold")
ax.set_ylabel("Record Count", fontsize=12, fontweight="bold")
ax.set_title("Gold Layer - Record Counts by Table", fontsize=14, fontweight="bold")
ax.set_xticklabels(tables, rotation=45, ha="right")
ax.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bar, val in zip(bars, count_vals):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f"{val:,}",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

## 3. OHLCV Bars - Sample Data

In [ ]:
# Query sample OHLCV data
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    ticker,
    timestamp,
    open,
    high,
    low,
    close,
    volume,
    vwap
FROM gold.ohlcv_bars
ORDER BY timestamp DESC, ticker
LIMIT 10
"""

df_ohlcv_sample = pd.read_sql(query, conn)
conn.close()

print("\nOHLCV Bars - Latest 10 Records:")
print("=" * 100)
display(df_ohlcv_sample)

## 4. OHLCV Bars - Data Coverage by Ticker

In [ ]:
# Query data coverage per ticker
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    ticker,
    COUNT(*) as record_count,
    MIN(date) as first_date,
    MAX(date) as last_date,
    ROUND(AVG(volume), 0) as avg_volume,
    ROUND(AVG(close), 2) as avg_close
FROM gold.ohlcv_bars
GROUP BY ticker
ORDER BY ticker
"""

df_coverage = pd.read_sql(query, conn)
conn.close()

print("\nOHLCV Data Coverage by Ticker:")
print("=" * 120)
display(df_coverage)

## 5. Macro Indicators - Sample Data

In [ ]:
# Query sample macro indicators
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    indicator_name,
    date,
    value,
    indicator_category,
    value_change,
    value_pct_change
FROM gold.macro_indicators
ORDER BY date DESC, indicator_name
LIMIT 10
"""

df_macro_sample = pd.read_sql(query, conn)
conn.close()

print("\nMacro Indicators - Latest 10 Records:")
print("=" * 120)
display(df_macro_sample)

## 6. Macro Indicators - Coverage by Indicator

In [ ]:
# Query macro indicator coverage
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    indicator_name,
    indicator_category,
    COUNT(*) as record_count,
    MIN(date) as first_date,
    MAX(date) as last_date,
    ROUND(AVG(value), 2) as avg_value
FROM gold.macro_indicators
GROUP BY indicator_name, indicator_category
ORDER BY indicator_name
"""

df_macro_coverage = pd.read_sql(query, conn)
conn.close()

print("\nMacro Indicators Coverage:")
print("=" * 120)
display(df_macro_coverage)

## 7. SEC Filings - Sample Data

In [ ]:
# Query sample SEC filings
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    ticker,
    company_name,
    form_type,
    filing_date,
    report_date,
    filing_category,
    accession_number
FROM gold.sec_filings
ORDER BY filing_date DESC
LIMIT 10
"""

df_filings_sample = pd.read_sql(query, conn)
conn.close()

print("\nSEC Filings - Latest 10 Records:")
print("=" * 120)
display(df_filings_sample)

## 8. SEC Filings - Breakdown by Type and Company

In [ ]:
# Query filings breakdown
conn = psycopg2.connect(**DB_CONFIG)

query = """
SELECT 
    ticker,
    form_type,
    COUNT(*) as filing_count,
    MIN(filing_date) as first_filing,
    MAX(filing_date) as last_filing
FROM gold.sec_filings
GROUP BY ticker, form_type
ORDER BY ticker, form_type
"""

df_filings_breakdown = pd.read_sql(query, conn)
conn.close()

print("\nSEC Filings Breakdown:")
print("=" * 100)
display(df_filings_breakdown)

## 9. Corporate Actions - Check for Data

In [ ]:
# Query corporate actions
conn = psycopg2.connect(**DB_CONFIG)

query = "SELECT * FROM gold.corporate_actions LIMIT 10"

df_corporate = pd.read_sql(query, conn)
conn.close()

if len(df_corporate) == 0:
    print("\nCorporate Actions: NO DATA")
    print("Expected: No corporate actions occurred in Q1 2026 for tracked tickers")
else:
    print("\nCorporate Actions - Sample:")
    print("=" * 100)
    display(df_corporate)

## 10. Summary Statistics

In [ ]:
# Generate summary statistics
conn = psycopg2.connect(**DB_CONFIG)

# OHLCV statistics
query_ohlcv = """
SELECT 
    COUNT(DISTINCT ticker) as unique_tickers,
    COUNT(DISTINCT date) as unique_dates,
    COUNT(*) as total_records,
    ROUND(AVG(volume), 0) as avg_daily_volume,
    ROUND(MIN(close), 2) as min_close,
    ROUND(MAX(close), 2) as max_close
FROM gold.ohlcv_bars
"""

df_stats_ohlcv = pd.read_sql(query_ohlcv, conn)

# Macro statistics
query_macro = """
SELECT 
    COUNT(DISTINCT indicator_name) as unique_indicators,
    COUNT(DISTINCT date) as unique_dates,
    COUNT(*) as total_records,
    MIN(date) as earliest_date,
    MAX(date) as latest_date
FROM gold.macro_indicators
"""

df_stats_macro = pd.read_sql(query_macro, conn)

# SEC filings statistics
query_filings = """
SELECT 
    COUNT(DISTINCT ticker) as unique_companies,
    COUNT(DISTINCT form_type) as unique_form_types,
    COUNT(*) as total_filings,
    MIN(filing_date) as earliest_filing,
    MAX(filing_date) as latest_filing
FROM gold.sec_filings
"""

df_stats_filings = pd.read_sql(query_filings, conn)

conn.close()

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)

print("\nOHLCV Bars:")
print("-" * 80)
display(df_stats_ohlcv)

print("\nMacro Indicators:")
print("-" * 80)
display(df_stats_macro)

print("\nSEC Filings:")
print("-" * 80)
display(df_stats_filings)

print("\n" + "=" * 80)

## 11. Data Quality Checks

In [ ]:
# Check for duplicates in OHLCV
conn = psycopg2.connect(**DB_CONFIG)

query_duplicates = """
SELECT ticker, timestamp, COUNT(*)
FROM gold.ohlcv_bars
GROUP BY ticker, timestamp
HAVING COUNT(*) > 1
"""

df_duplicates = pd.read_sql(query_duplicates, conn)

# Check for NULL values in critical columns
query_nulls = """
SELECT 
    SUM(CASE WHEN ticker IS NULL THEN 1 ELSE 0 END) as null_tickers,
    SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END) as null_close,
    SUM(CASE WHEN volume IS NULL THEN 1 ELSE 0 END) as null_volume
FROM gold.ohlcv_bars
"""

df_nulls = pd.read_sql(query_nulls, conn)
conn.close()

print("\nData Quality Checks:")
print("=" * 80)

if len(df_duplicates) == 0:
    print("PASS No duplicates found in OHLCV data")
else:
    print(f"FAIL Found {len(df_duplicates)} duplicate records:")
    display(df_duplicates)

print("\nNULL Value Check:")
display(df_nulls)

if df_nulls.iloc[0].sum() == 0:
    print("PASS No NULL values in critical columns")
else:
    print("FAIL NULL values detected in critical columns")